# Aivora AI - Continue financial_poc Training on Google Colab

This notebook continues the **same** `aivora-ai-financial-poc-round-2` run
that also runs on Kaggle - it is not a separate/smaller training job. It
reproduces `configs/financial_poc.yaml`'s real dataset mix and token
budgets, requires a real checkpoint to resume from (it will not silently
start over from step 0), and calls the exact same `training.trainer.train_model`
function the Kaggle notebook and `main.py train` call locally.

Nothing here is simulated: every cell either does the real thing or raises
`RuntimeError('STATUS = BLOCKED: ...')` naming exactly what's missing, the
same discipline used in `training/kaggle/Aivora_Kaggle_Training.ipynb`.

**Before running:** Runtime -> Change runtime type -> Hardware accelerator -> GPU.


## 1. Environment verification

In [ ]:
import sys, platform, os
print('python:', sys.version)
print('platform:', platform.platform())
print('cwd:', os.getcwd())


## 2-5. GPU, CUDA, PyTorch, and VRAM verification (hard gate)

If `torch.cuda.is_available()` is not `True`, STOP - do not proceed to claim GPU training happened.

In [ ]:
!nvidia-smi


In [ ]:
import torch

gpu_available = torch.cuda.is_available()
print('torch:', torch.__version__)
print('torch cuda build:', torch.version.cuda)
print('GPU AVAILABLE =', gpu_available)

if not gpu_available:
    raise RuntimeError(
        'STATUS = BLOCKED: torch.cuda.is_available() is False. '
        'Set Runtime -> Change runtime type -> GPU, then re-run from the top.'
    )

device_count = torch.cuda.device_count()
props = torch.cuda.get_device_properties(0)
vram_gb = round(props.total_memory / 1024**3, 2)

print('GPU name:', props.name)
print('device count:', device_count)
print('compute capability:', f'{props.major}.{props.minor}')
print('total VRAM GB:', vram_gb)

GPU_NAME = props.name
VRAM_GB = vram_gb


## 6. Repository acquisition

Clones the same GitHub repo the Kaggle run uses, so this session has the
exact same `training/trainer.py`, `training/checkpoint_utils.py`,
`training/notifier_utils.py`, and `configs/financial_poc.yaml` - not a
locally-edited or stale copy.

In [ ]:
# OPTION A: clone the real repo (default - this is a public HTTPS clone,
# same as the Kaggle notebook uses, so no credentials are needed here).
!git clone --depth 1 https://github.com/Ankushk-aosc/Aivora-AI.git
%cd Aivora-AI

# OPTION B (fallback): upload the repo as a zip via the Colab file browser
# (left sidebar) instead, then comment out Option A above and uncomment:
# !unzip -q Aivora-AI.zip
# %cd Aivora-AI


In [ ]:
# Repository integrity check - confirm the critical paths actually came
# across, including the checkpoint/notifier utilities this notebook needs.
import os
required = [
    'models/model.py', 'training/trainer.py', 'training/checkpoint_utils.py',
    'training/notifier_utils.py', 'ai_platform/model_registry.py',
    'data_sources/prepare.py', 'evaluation/evaluator.py',
    'configs/financial_poc.yaml', 'inference/generator.py',
]
missing = [p for p in required if not os.path.exists(p)]
if missing:
    raise RuntimeError(f'STATUS = BLOCKED: repo clone incomplete, missing {missing}')
print('Repository integrity check passed:', len(required), 'required paths present.')


## 7. Dependency installation

In [ ]:
# Colab ships a CUDA-enabled torch already - do NOT reinstall the CPU
# wheel from requirements.txt over it. `kaggle` is only needed if you use
# Checkpoint transfer Option C below.
!pip install -q tiktoken datasets pyyaml pypdf python-docx scikit-learn networkx langdetect kaggle
!python -m pip check


In [ ]:
import torch, tiktoken, datasets, yaml, sklearn, networkx, langdetect
print('All required imports succeeded')
print('torch cuda still available after installs:', torch.cuda.is_available())


## 8. Checkpoint transfer

This run must resume from a real checkpoint - it does not start a fresh,
smaller training job. Get the latest checkpoint (`.pt` + its `.json`
sidecar, e.g. `checkpoint_16000.pt`/`.json`) into `checkpoints/base/` in
THIS session using whichever of the three options below fits you; only one
is needed.

- **Option A - manual upload:** open the Colab file browser (left sidebar,
  folder icon), navigate into `Aivora-AI/checkpoints/base/`, and drag both
  files in from your machine.
- **Option B - Google Drive:** if you've saved the checkpoint to Drive,
  mount it and copy the two files in (cell below, commented out).
- **Option C - download from the Kaggle checkpoint dataset:** requires your
  own Kaggle API credentials added to Colab's Secrets panel (left sidebar,
  key icon) as `KAGGLE_USERNAME` and `KAGGLE_KEY` - add those secrets
  yourself; this notebook never asks for or stores them directly (cell
  below, commented out).

In [ ]:
# Option B - Google Drive (uncomment and edit the source path to match
# where you saved the checkpoint in your Drive):
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p checkpoints/base
# !cp "/content/drive/MyDrive/<path-to>/checkpoint_16000.pt" checkpoints/base/
# !cp "/content/drive/MyDrive/<path-to>/checkpoint_16000.json" checkpoints/base/

# Option C - Kaggle download (uncomment; requires KAGGLE_USERNAME/KAGGLE_KEY
# already added under Colab's Secrets panel):
# from google.colab import userdata
# os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
# os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
# !mkdir -p checkpoints/base
# !kaggle datasets download aoscjkjhh/aivora-ai-financial-poc-checkpoint -p checkpoints/base --unzip

print('Uncomment ONE option above (or use Option A via the file browser) before continuing.')


## 9. Dataset preparation

Reproduces `configs/financial_poc.yaml`'s real `dataset_mix` and
`dataset_token_overrides` exactly as the Kaggle run does - not a smaller
or different ad hoc budget. Streamed live from Hugging Face with a hard
token budget per dataset, using the same verified sources in
`data_sources/dataset_registry.py`.

In [ ]:
PRESET = 'financial_poc'

import yaml
from data_sources.dataset_registry import list_entries
from data_sources.dataset_mixer import BUCKET_TO_CATEGORY, validate_mix
from data_sources.prepare import prepare_dataset

with open(f'configs/{PRESET}.yaml') as f:
    preset_cfg = yaml.safe_load(f)

mix = preset_cfg['dataset_mix']
validate_mix(mix)
overrides = preset_cfg.get('dataset_token_overrides', {})
total_budget = int(preset_cfg['train_tokens']) + int(preset_cfg['validation_tokens'])
print(f"Preset '{PRESET}': total token budget {total_budget:,} across {len(mix)} buckets")
if overrides:
    print(f'  ({len(overrides)} dataset(s) use an explicit dataset_token_overrides ceiling)')

summary = []
for bucket, weight in mix.items():
    category = BUCKET_TO_CATEGORY[bucket]
    entries = list_entries(category=category, verified_only=True)
    if not entries:
        print(f"  [SKIP] bucket '{bucket}' (category '{category}') has no VERIFIED datasets registered.")
        continue
    auto_entries = [e for e in entries if e.name not in overrides]
    bucket_budget = int(total_budget * weight)
    auto_per_dataset_budget = max(bucket_budget // len(auto_entries), 50_000) if auto_entries else 0
    for entry in entries:
        per_dataset_budget = overrides.get(entry.name, auto_per_dataset_budget)
        print(f"  Preparing '{entry.name}' (bucket '{bucket}', budget {per_dataset_budget:,} tokens)...")
        result = prepare_dataset(entry.name, max_tokens=per_dataset_budget)
        summary.append((entry.name, result['train_tokens_used'], result['validation_tokens_used']))

print()
print('Prepared datasets (real, measured token counts):')
for name, train_tok, val_tok in summary:
    print(f'  {name}: {train_tok:,} train / {val_tok:,} validation tokens')


## 10. Leakage check (hard gate)

Confirms no evaluation question appears verbatim in the training shards just prepared.

In [ ]:
from evaluation import check_leakage

leak_report = check_leakage()
print(leak_report)
if not leak_report.get('clean', False):
    raise RuntimeError(f'STATUS = BLOCKED: leakage detected - {leak_report}')
print('Leakage check passed: no evaluation text found in training shards.')


## 11. Tokenizer round-trip check

In [ ]:
from data_sources.tokenizer import get_encoding
enc = get_encoding()
probe = enc.encode_ordinary('What is EBITDA margin?')
print('vocab size:', enc.n_vocab)
assert enc.decode(probe) == 'What is EBITDA margin?'
print('Tokenizer round-trip: OK')


## 12. Model configuration

In [ ]:
from models import DeepSeekConfig, DeepSeekV3

config = DeepSeekConfig.default()
model_preview = DeepSeekV3(config)
param_count = sum(p.numel() for p in model_preview.parameters())
print(f'Model: {param_count:,} parameters')
del model_preview
torch.cuda.empty_cache()


## 13. Checkpoint discovery + compatibility check (hard gate)

Uses `training.checkpoint_utils` (the same module the Kaggle notebook's
resume logic is built on) to find the highest-step checkpoint under
`checkpoints/` and verify it's architecture-compatible with the current
model before trusting it. Fails clearly if step 8 wasn't completed.

In [ ]:
from training.checkpoint_utils import resolve_resume_checkpoint

ref_model = DeepSeekV3(DeepSeekConfig.default())
RESUME_CHECKPOINT = resolve_resume_checkpoint('checkpoints', ref_model, required=True)
del ref_model
torch.cuda.empty_cache()

ckpt_step = torch.load(RESUME_CHECKPOINT, map_location='cpu').get('step')
print(f'Resuming from {RESUME_CHECKPOINT} (step {ckpt_step}).')


## 14. Training

Resumes from `RESUME_CHECKPOINT` using `financial_poc.yaml`'s own tuned
`batch_size`/`seq_len`/`gradient_accumulation_steps` unmodified - these
values are NOT re-derived from this GPU's VRAM, because the checkpoint's
optimizer state (Adam momentum/variance) was tuned under that exact
effective batch size; overriding it here would invalidate that state (see
`configs/financial_poc.yaml`'s own notes on this). If CUDA OOM occurs, the
retry loop below shrinks batch size/seq_len and retries, same as the
Kaggle notebook. Checkpoints save every `eval_interval` steps and email
notifications fire automatically (via `training/notifier_utils.py`) if you
attached `NOTIFIER_EMAIL_ADDRESS`/`NOTIFIER_EMAIL_PASSWORD` under Colab's
Secrets panel (left sidebar, key icon).

In [ ]:
import yaml as _yaml
from training.trainer import train_model

CFG_PATH = f'configs/{PRESET}.yaml'


def attempt_training(resume, max_retries=4):
    attempt = 0
    while attempt < max_retries:
        try:
            return train_model(preset_name=PRESET, resume=resume)
        except torch.cuda.OutOfMemoryError as e:
            attempt += 1
            print(f'CUDA OOM on attempt {attempt}/{max_retries}: {e}')
            torch.cuda.empty_cache()
            if attempt >= max_retries:
                raise RuntimeError(
                    f'STATUS = BLOCKED: repeated CUDA OOM after {max_retries} attempts. '
                    f'See configs/{PRESET}.yaml to reduce batch_size/seq_len by hand.'
                )
            with open(CFG_PATH) as f:
                cfg = _yaml.safe_load(f)
            if cfg['batch_size'] > 2:
                cfg['batch_size'] = max(2, cfg['batch_size'] // 2)
                print(f"Reducing batch_size to {cfg['batch_size']} and retrying.")
            elif cfg.get('seq_len') and cfg['seq_len'] > 128:
                cfg['seq_len'] = max(128, cfg['seq_len'] // 2)
                print(f"Reducing seq_len to {cfg['seq_len']} and retrying.")
            else:
                raise RuntimeError(
                    'STATUS = BLOCKED: batch_size/seq_len are already at the minimum this notebook will try automatically.'
                )
            with open(CFG_PATH, 'w') as f:
                _yaml.dump(cfg, f)


model, config, ckpt_path = attempt_training(RESUME_CHECKPOINT)
print(f'Training completed. Final checkpoint: {ckpt_path}')


## 15. Evaluation

In [ ]:
from evaluation import evaluate_model, print_report

eval_results = evaluate_model(model, device='cuda', max_new_tokens=48, verbose=True)
print_report(eval_results)


## 16. Inference test (the 3 required prompts)

In [ ]:
from inference import load_model_for_inference, generate_text

test_prompts = [
    'What is EBITDA?',
    'Calculate EBITDA margin for revenue 500 and EBITDA 100.',
    'What is working capital?',
]
for prompt in test_prompts:
    output = generate_text(ckpt_path, prompt, max_tokens=60, temperature=0.7, top_k=40, device='cuda')
    print(f'PROMPT: {prompt}')
    print(f'OUTPUT: {output}')
    print('-' * 60)


## 17. Checkpoint export + hash verification

In [ ]:
import json
from ai_platform.model_registry import register_checkpoint, verify_integrity

entry = register_checkpoint(ckpt_path, stage='base', set_active=True)
print('Registered checkpoint:', entry)

verification = verify_integrity(entry['version'])
print('Integrity verification:', verification)
if not verification['valid']:
    raise RuntimeError(f'STATUS = BLOCKED: checkpoint integrity verification failed: {verification}')

manifest = {
    'preset': PRESET,
    'checkpoint_path': ckpt_path,
    'gpu': GPU_NAME,
    'registry_entry': entry,
}
with open('export_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2, default=str)
print('Wrote export_manifest.json')


---
## 18. Bringing the checkpoint back to your local project

1. In the Colab file browser (left sidebar), download the checkpoint
   `.pt` + `.json` pair from `checkpoints/base/` (and `export_manifest.json`).
2. Place them in your local repo under `checkpoints/base/`.
3. Register it locally:
   ```bash
   python -c "from ai_platform.model_registry import register_checkpoint; print(register_checkpoint('checkpoints/base/<checkpoint_name>.pt', stage='base'))"
   ```
4. If you also want this progress reflected in the Kaggle checkpoint
   dataset (so the next Kaggle resume picks up from here instead of
   restarting from an older step), upload the new checkpoint as a new
   version of the `aivora-ai-financial-poc-checkpoint` Kaggle Dataset.
5. Restart the backend pointed at the new checkpoint and re-run
   `python ai_platform/acceptance_test.py` against it before considering
   this progress verified.